## Ejercicio 1: Evaluación de solicitud de crédito bancario
 
Aquí el agente recibe los datos de una solicitud de crédito y tiene que decidir solo, sin que
intervenga un asesor, si el préstamo se aprueba o no.
 
### Ficha PEAS
 
| | |
|---|---|
| **Percepción (S)** | El ingreso mensual del cliente, el monto que está pidiendo y si tiene o no historial moroso. Con los dos primeros datos el agente calcula la relación monto/ingreso, que es lo que realmente usa para decidir. |
| **Acciones (A)** | Aprobar, aprobar con condiciones (pedir garante o bajar el monto) o rechazar. |
| **Entorno (E)** | El área de créditos del banco: las solicitudes que van llegando, la base de datos donde está el historial de cada cliente y las políticas de riesgo que maneja la entidad. |
| **Objetivo** | Prestar plata a quien sí va a poder pagarla y no prestársela a quien no. |
| **Medida de desempeño (P)** | Cuántos créditos aprobados terminan pagándose sin atraso, cuánta morosidad acumula el banco y cuántos clientes buenos se rechazaron por gusto. |
 
**Tipo de agente:** es un agente reactivo simple. Funciona con reglas del tipo si-entonces y decide
solo con los datos de la solicitud que tiene al frente; no se acuerda de las solicitudes anteriores ni
aprende de ellas.
 
**¿Es racional?** Sí, siempre que los datos que recibe sean correctos. Si el ingreso llegara mal
registrado (por ejemplo un ingreso de 0 o negativo), el agente estaría decidiendo sobre un cliente que
no existe, así que la función revisa eso antes de calcular nada.
 

### Justificación de las reglas
 
Elegí la relación monto/ingreso porque es lo que de verdad mide si el cliente puede pagar: no es lo
mismo pedir S/ 2000 ganando S/ 4000 que pedir lo mismo ganando S/ 800. Mientras más sube esa
relación, más se le complica la cuota al cliente, por eso separé la decisión en tramos en vez de usar
un solo corte. El historial moroso lo tomé como agravante y le bajé los umbrales a la mitad, porque
alguien que ya dejó de pagar antes tiene más chance de volver a hacerlo. Dejé el tramo intermedio de
"aprobar con condiciones" para no perder clientes que sí sirven y solo necesitan un monto más bajo o
un plazo más largo, que al banco le conviene más que decirles que no.
 
**Reglas:**
 
| Historial moroso | Relación monto/ingreso | Decisión |
|---|---|---|
| No | hasta 1.0 | aprobar |
| No | entre 1.0 y 2.0 | aprobar con condiciones |
| No | más de 2.0 | rechazar |
| Sí | hasta 0.5 | aprobar con condiciones |
| Sí | más de 0.5 | rechazar |
 
Cuando la relación cae justo en el umbral (1.0, 2.0 o 0.5), se queda en el tramo menos estricto.
 

### Código
 

In [7]:
import numpy as np
import matplotlib.pyplot as plt
 
 
def agente_credito(ingreso_mensual, monto_solicitado, tiene_historial_moroso):
    # si el ingreso llega en 0 o negativo el dato esta mal y ademas no se puede dividir
    if ingreso_mensual <= 0:
        return "rechazar", "el ingreso reportado no es valido, no se puede evaluar"
 
    relacion = monto_solicitado / ingreso_mensual
    r = round(relacion, 2)
 
    # cliente con historial moroso: se le exige el doble
    if tiene_historial_moroso:
        if relacion <= 0.5:
            return "aprobar con condiciones", "relacion de " + str(r) + " es baja pero tiene historial moroso, se pide garante"
        else:
            return "rechazar", "relacion de " + str(r) + " mayor a 0.5 con historial moroso"
 
    # cliente sin historial moroso
    if relacion <= 1.0:
        return "aprobar", "relacion de " + str(r) + " menor o igual a 1.0 y sin historial moroso"
    elif relacion <= 2.0:
        return "aprobar con condiciones", "relacion de " + str(r) + " entre 1.0 y 2.0, conviene bajar el monto o alargar el plazo"
    else:
        return "rechazar", "relacion de " + str(r) + " mayor a 2.0, el monto es muy alto para su ingreso"